In [ ]:
import csv, hashlib, json, shutil, subprocess, sys
from pathlib import Path

def run(*args):
    subprocess.run(list(map(str, args)), check=True)
def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
def only(name):
    candidates = list(Path('/kaggle/input').rglob(name))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {name}: {candidates}')
    return candidates[0]

work = Path('/kaggle/temp/t78_readme')
work.mkdir(parents=True, exist_ok=False)
repo = work / 'repo'
run('git', 'clone', '--quiet', '--branch', 'handoff-repro-checklist', 'https://github.com/ttt-huong/helmet-detection-yolov8.git', repo)
commit = 'e04187e07fb75f053dbe0ba355b84e4302385780'
run('git', '-C', repo, 'checkout', '--quiet', '--detach', commit)
run(sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.202', 'gdown')
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
print('README commit:', commit, '| GPU:', torch.cuda.get_device_name(0), flush=True)
archive = work / 'VOC2028.zip'
run(sys.executable, '-m', 'gdown', '1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX', '-O', archive)
assert sha(archive) == '2f6c5c347b778271bc8b27c2d61b731fc5fb8cdb4f45fce583dd2f82e6a46401'
root = work / 'helmet_yolo'
run(sys.executable, repo/'scripts/prepare_shwd.py', '--archive', archive, '--manifests', repo/'manifests', '--output', root)
source = {key: only(key + '_best.pt') for key in ('v8n', 'v8s', 'v8m')}
baseline = only('kaggle_resume_metadata.json')
assert json.loads(baseline.read_text())['completed_epochs'] == 30
source['v5su'] = baseline.parent / 'weights/best.pt'
reference = list(csv.DictReader((repo/'results/helmet_test_summary.csv').open()))
ref_by_key = {(r['model'], r['split']): r for r in reference}
weight_args = []
for key, path in source.items():
    assert sha(path) == ref_by_key[key,'test_clean']['checkpoint_sha256'], key
    out = work / 'weights' / key / 'best.pt'
    out.parent.mkdir(parents=True)
    shutil.copyfile(path, out)
    weight_args.extend(('--'+key, out))
from ultralytics import YOLO
model = YOLO(str(work/'weights/v8n/best.pt'))
image = Path((root/'test_clean.txt').read_text().splitlines()[0])
assert model.predict(str(image), device=0, imgsz=640, verbose=False)[0].orig_shape
print('Single-image inference per README passed', flush=True)
del model
output = Path('/kaggle/working/helmet_t78_reproduction')
run(sys.executable, '-u', repo/'scripts/evaluate_helmet.py', '--data-root', root, '--manifests', repo/'manifests', '--output', output, *weight_args)
actual = list(csv.DictReader((output/'summary.csv').open()))
assert {(r['model'],r['split']) for r in actual} == set(ref_by_key) and len(actual) == 12
assert all(r['images'] == ref_by_key[r['model'],r['split']]['images'] for r in actual)
metrics = ('precision','recall','map50','map50_95')
deltas = [{'model':r['model'], 'split':r['split'], **{m:float(r[m])-float(ref_by_key[r['model'],r['split']][m]) for m in metrics}} for r in actual]
protocol = json.loads((output/'protocol.json').read_text())
assert protocol['status'] == 'complete' and 'T4' in protocol['gpu']
assert len(list(csv.DictReader((output/'speed_raw.csv').open()))) == 1200
assert len(list(csv.DictReader((output/'speed_summary.csv').open()))) == 4
result = {'readme_commit':commit,'gpu':protocol['gpu'],'eval_count':12,'timing_count':1200,'max_absolute_delta':{m:max(abs(r[m]) for r in deltas) for m in metrics},'per_run_delta':deltas}
(output/'reproduction_check.json').write_text(json.dumps(result, indent=2))
print('T78 VERIFIED:', json.dumps(result['max_absolute_delta']), flush=True)
